<a href="https://colab.research.google.com/github/Adriana-Sousa/machine-learning/blob/main/processamento_de_linguagem_natural/pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 1.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


In [2]:
from gtts import gTTS

text_to_say = "How are you doing?."

language = "en"

gtts_object = gTTS(text = text_to_say,
                  lang = language,
                  slow = False)

gtts_object.save("/content/gtts.wav")

In [3]:
from IPython.display import Audio

Audio("/content/gtts.wav")

In [4]:
# Instalar dependências
!pip install SpeechRecognition pyjokes wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.0 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=c90000aadb7c008d08e9870ff6317db2babe154e27bb7dbab5074377c687181b
  Stored in directory: /root/.cache/pip/wheels/8f/ab/cb/45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia


In [5]:
import os
import io
import base64
import pyjokes
import wikipedia
from gtts import gTTS
from datetime import datetime
import speech_recognition as sr
from IPython.display import Audio, display
from google.colab import output

In [ ]:
# Função para gravar áudio via navegador
def record_audio_colab():
    js = """
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let data = [];

      recorder.ondataavailable = event => data.push(event.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, 5000)); // grava 5 segundos
      recorder.stop();

      await new Promise(resolve => recorder.onstop = resolve);
      const audioBlob = new Blob(data, { type: 'audio/wav' });
      const reader = new FileReader();
      reader.readAsDataURL(audioBlob);
      reader.onloadend = () => {
        const base64data = reader.result.split(',')[1];
        google.colab.kernel.invokeFunction('notebook.get_audio', [base64data], {});
      };
    }
    recordAudio();
    """
    display(output.eval_js(js))

In [ ]:
# Função para receber áudio do JavaScript
def get_audio_from_js(b64):
    binary = base64.b64decode(b64)
    with open("audio.wav", "wb") as f:
        f.write(binary)

output.register_callback('notebook.get_audio', get_audio_from_js)

In [ ]:
# Função para falar
def speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("voice.mp3")
    display(Audio("voice.mp3", autoplay=True))

In [ ]:
# Função para converter áudio em texto
def get_audio():
    print("Gravando... fale algo!")
    record_audio_colab()

    import time
    time.sleep(7)  # Espera gravação + salvamento

    if not os.path.exists("audio.wav"):
        speak("Não consegui capturar o áudio.")
        return ""

    r = sr.Recognizer()
    with sr.AudioFile("audio.wav") as source:
        audio = r.record(source)
    try:
        said = r.recognize_google(audio)
        print("Você disse:", said)
        return said.lower()
    except:
        speak("Não consegui entender.")
        return ""

In [ ]:
Audio("/content/audio.wav")

In [ ]:
# Função de resposta
def respond(text):
    if 'youtube' in text:
        speak("What do you want to search for?")
        keyword = get_audio()
        if keyword:
            print(f"https://www.youtube.com/results?search_query={keyword}")
    elif 'search' in text:
        speak("What do you want to search for?")
        query = get_audio()
        if query:
            result = wikipedia.summary(query, sentences=3)
            speak("According to Wikipedia")
            print(result)
    elif 'joke' in text:
        speak(pyjokes.get_joke())
    elif 'what time' in text:
        strTime = datetime.now().strftime("%H:%M %p")
        speak(f"The time is {strTime}")
    elif 'exit' in text:
        speak("Goodbye!")
        raise SystemExit

In [ ]:
print("Diga um comando...")
text = get_audio()
respond(text)